# Open-Source CMOS PTAT Temperature Sensor with Digital Calibration

**ISSCC 2027 Code-a-Chip**

This notebook is the primary reproducible artifact for a SKY130 PTAT temperature
sensor study. It separates analytical, retained real-PDK, synthetic, and
unavailable evidence so simulation results are not presented as silicon
measurements.

The core idea uses the weak-inversion relation ΔVGS ≈ n·kT/q·ln(N), with an
effective current-density ratio N=8, followed by digital calibration.

**GitHub submission contact:** [@3more102](https://github.com/3more102)


## Submission identity and verified tool versions

- Submission contact: **@3more102** (GitHub repository owner/contact).
- Primary artifact: this Jupyter notebook, released under Apache-2.0 with the project files.
- Retained transistor evidence: **ngspice 46**.
- Retained open PDK provenance: **open_pdks/SKY130 `12df12e2e74145e31c5a13de02f9a1e176b56e67`**.
- Repository CI executes the notebook and Python checks on **Python 3.12**.

The retained run-221 manifest is the source of truth for the SPICE/PDK versions; the CI Python version describes current reproducibility verification, not the historical transistor run.


## Evidence contract

The retained real-PDK evidence was generated with ngspice and SKY130/open_pdks.
The repository preserves raw CSV data and a machine-readable manifest.

The release calibration is fixed five-point PWL at **−40, −20, 0, 50, 125 °C**.
Endpoint two-point calibration is included as a disclosed negative result because
it does not meet the internal 0.5 °C target.

In [ ]:
from pathlib import Path
import json, math
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "results").is_dir():
    ROOT = ROOT / "ISSCC27" / "submitted_notebooks" / "ptat_temp_sensor"

design = json.loads((ROOT / "design_requirements.json").read_text())
manifest = json.loads((ROOT / "results" / "sky130_ci_manifest.json").read_text())
cal = json.loads((ROOT / "results" / "pdk_calibration_analysis.json").read_text())
release = json.loads((ROOT / "release_requirements.json").read_text())
print("PDK:", manifest["pdk_sources"])
print("Corners:", manifest["corners"])
print("Retained temperatures:", manifest["temperature_c"])

## Analytical PTAT expectation

The analytical model provides intuition and a cross-check, not a replacement for
transistor-level evidence.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "src"))
from ptat_model import PTATParams, nominal_slope_v_per_k

p = PTATParams(current_density_ratio=design["current_density_ratio"])
print(f"Analytical default slope: {1e6*nominal_slope_v_per_k(p):.3f} uV/K")
for c in ("tt","ff","ss"):
    s = manifest["summary"]["mirror"][c]["ptat_slope_uv_per_k"]
    print(f"Retained mirror {c.upper()}: {s:.3f} uV/K")

## Retained real-PDK temperature response

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for corner in ("tt","ff","ss"):
    df = pd.read_csv(ROOT / "results" / "sky130_ci" / f"ptat_mirror_{corner}.csv",
                     skipinitialspace=True)
    ax.plot(df["temp_c"], 1e3*df["dvgs_v"], marker="o", label=corner.upper())
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("ΔVGS (mV)")
ax.set_title("Retained SKY130 PMOS-mirror PTAT response")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## Calibration result

Two-point endpoint calibration misses the internal target. The five-point PWL
architecture is therefore the release method rather than hiding the miss.

In [ ]:
two = cal["two_point_endpoint_calibration"]["worst_max_abs_error_c"]
pwl = cal["five_point_pwl_grid_calibration"]["worst_max_abs_error_c"]
anchors = cal["five_point_pwl_grid_calibration"]["anchors_c"]
print(f"Two-point worst retained sampled error: {two:.3f} °C")
print(f"Five-point PWL anchors: {anchors}")
print(f"Five-point PWL worst retained sampled error: {pwl:.3f} °C")
assert two > 0.5
assert pwl <= release["release_targets"]["sampled_grid_pwl_max_abs_error_c_max"]

## Calibration error by retained dataset

In [ ]:
rows=[]
for name,m in cal["five_point_pwl_grid_calibration"]["per_dataset"].items():
    rows.append({"dataset":name,
                 "max_abs_error_c":m["max_abs_error_c"],
                 "rms_error_c":m["rms_error_c"]})
pd.DataFrame(rows).sort_values("max_abs_error_c", ascending=False)

## Behavioral ADC/readout budget

The project does not claim a transistor-level ADC. The release model uses a
12-bit, 1.8-V ADC with analog gain 10, selected to preserve headroom and keep the
quantized retained-grid calibration within 0.5 °C.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("readout_budget", ROOT/"readout_budget.py")
rb = importlib.util.module_from_spec(spec); spec.loader.exec_module(rb)
readout = rb.analyze()
print(json.dumps({
    "status":readout["status"],
    "gain":readout["analog_gain"],
    "worst_quantization_rms_c":readout["worst_quantization_rms_c"],
    "worst_full_scale_utilization":readout["worst_full_scale_utilization"],
    "worst_quantized_pwl_sampled_error_c":readout["worst_quantized_pwl_sampled_error_c"],
}, indent=2))
assert readout["status"] == "PASS"

## Power and mismatch boundaries

The retained PMOS-mirror testbench includes the PTAT sensing branches and PMOS
bias network, but the reference current source is ideal and the ADC is
behavioral. Therefore this project reports **core/testbench power only**, not
whole-chip power.

The retained TT/FF/SS branch-current mismatch is deterministic corner behavior,
not statistical local mismatch. A real SKY130 tt_mm Monte-Carlo harness is
included as mismatch_mc.py; results are promoted only after a successful retained
run with the pinned PDK.

In [ ]:
summary = manifest["summary"]["mirror"]
print("Worst retained mirror power (uW):",
      max(v["max_power_uw"] for v in summary.values()))
print("Worst retained deterministic branch mismatch (%):",
      max(v["max_branch_mismatch_percent"] for v in summary.values()))

## Dense-grid and physical verification

The retained run has eight temperature samples. run_sky130.py and
dense_characterization.py provide a new real-PDK dense sweep that scores only
actual simulated temperatures, with no interpolated error claims.

A final layout is encouraged by Code-a-Chip but not required. The layout
directory defines the intended LVS boundary and evidence contract. DRC/LVS/PEX
remain **not claimed** until retained artifacts exist.

## Reproducibility

From this project directory run:

    python evidence_audit.py
    python pdk_calibration_analysis.py --check
    python readout_budget.py --check
    python submission_preflight.py

With ngspice and the pinned SKY130/open_pdks PDK installed:

    python run_sky130.py --mode both --corners tt ff ss --temps=-40:125:5
    python dense_characterization.py
    python mismatch_mc.py --samples 100 --temps=-40:125:5

The notebook remains reproducible without a local PDK installation because the
original run-221 CSV evidence and provenance are retained in the repository.

## References

1. IEEE SSCS Open-Source Ecosystem, **Code-a-Chip / ISSCC 2027** submission repository and rules: https://github.com/sscs-ose/sscs-ose-code-a-chip.github.io
2. OpenFASoC, **Temperature Sensor Generator** notebook/example: https://github.com/idea-fasoc/OpenFASOC/tree/main/docs/source/notebooks/temp-sense-gen
3. ngspice project documentation: https://ngspice.sourceforge.io/
4. SKY130 open PDK ecosystem / open_pdks project: https://github.com/RTimothyEdwards/open_pdks

No numerical result in this notebook is copied from these references; project results are generated from the analytical model or the retained repository evidence identified above.


## Limitations

- No silicon measurements are available.
- No post-layout result is claimed.
- No statistical mismatch result is promoted until the new Monte-Carlo workflow
  successfully runs and its outputs are retained.
- The 0.472 °C result is an eight-sample retained-grid result, not a guarantee
  between those temperatures.

These limitations are part of the result, not hidden exceptions.